# MNIST CNN Filter and Complexity Lab

**Aim:** Explore how convolutional layers transform image structure, compare hand-designed and learned filters, calculate CNN complexity, and avoid common beginner mistakes.

Each section uses a short **what / why** explanation before the code.

## 1. Setup and reproducibility

**What:** Import the numerical, plotting, deep-learning, and evaluation tools and fix random seeds.
**Why:** Reproducibility makes filter visualizations and model comparisons easier to trust.

In [ ]:
import os
import random
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, Markdown
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")

## 2. Inspect MNIST before transformation

**What:** Load the official MNIST partitions and display ten examples.
**Why:** Inspecting raw dimensions and labels prevents silent shape and label assumptions.

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.mnist.load_data()
print(f"Raw train images: {x_train_raw.shape}; raw train labels: {y_train_raw.shape}")
print(f"Raw test images:  {x_test_raw.shape}; raw test labels:  {y_test_raw.shape}")
assert x_train_raw.ndim == x_test_raw.ndim == 3
assert y_train_raw.ndim == y_test_raw.ndim == 1
assert np.min(y_train_raw) >= 0 and np.max(y_train_raw) <= 9
assert np.min(y_test_raw) >= 0 and np.max(y_test_raw) <= 9
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_train_raw[:10], y_train_raw[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()

## 3. Leakage-safe preprocessing

**What:** Split raw training data first, then scale pixels and add a channel dimension.
**Why:** The test set remains untouched and every CNN input has explicit `(height, width, channels)` structure.

In [ ]:
x_train_raw, x_val_raw, y_train_raw, y_val_raw = train_test_split(
    x_train_raw, y_train_raw, test_size=0.10, stratify=y_train_raw, random_state=SEED
)

def prepare_images(images):
    images = images.astype("float32") / 255.0
    images = images[..., np.newaxis]
    return images

x_train = prepare_images(x_train_raw)
x_val = prepare_images(x_val_raw)
x_test = prepare_images(x_test_raw)
y_train, y_val, y_test = y_train_raw.astype("int64"), y_val_raw.astype("int64"), y_test_raw.astype("int64")
for name, images, labels in [("train", x_train, y_train), ("validation", x_val, y_val), ("test", x_test, y_test)]:
    assert images.shape[1:] == (28, 28, 1)
    assert images.dtype == np.float32
    assert np.isfinite(images).all() and 0.0 <= images.min() <= images.max() <= 1.0
    assert labels.ndim == 1 and labels.min() >= 0 and labels.max() <= 9
    print(f"{name:10s}: images {images.shape}, labels {labels.shape}, range [{images.min():.1f}, {images.max():.1f}]")
assert len(x_train) + len(x_val) == 60000
assert len(x_test) == 10000

## 4. Fixed filter exploration

**What:** Apply several hand-designed `3 x 3` kernels with a reusable user function.
**Why:** Different kernels emphasize edges, smoothness, contrast, or texture before a learned CNN is introduced.

In [ ]:
def apply_kernel(image, kernel):
    """Apply a 3x3 correlation kernel while preserving image dimensions."""
    image = np.asarray(image, dtype=np.float32)
    kernel = np.asarray(kernel, dtype=np.float32)
    assert image.ndim == 2 and kernel.shape == (3, 3)
    padded = np.pad(image, 1, mode="reflect")
    response = np.empty_like(image)
    for row in range(image.shape[0]):
        for col in range(image.shape[1]):
            response[row, col] = np.sum(padded[row:row + 3, col:col + 3] * kernel)
    return response

FILTERS = {
    "Horizontal Sobel": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),
    "Vertical Sobel": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
    "Laplacian": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32),
    "Sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
    "Box blur": np.ones((3, 3), dtype=np.float32) / 9,
    "Gaussian-like blur": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32) / 16,
    "Emboss": np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], dtype=np.float32),
}
source_image = x_train[0, ..., 0]
filter_responses = {name: apply_kernel(source_image, kernel) for name, kernel in FILTERS.items()}
assert all(response.shape == source_image.shape for response in filter_responses.values())
print("Filter response structure:", source_image.shape, "->", next(iter(filter_responses.values())).shape)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()
axes[0].imshow(source_image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original digit")
for ax, (name, response) in zip(axes[1:], filter_responses.items()):
    ax.imshow(response, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(name)
for ax in axes: ax.axis("off")
plt.tight_layout()

**Filter notes:** Sobel kernels detect directional edges; Laplacian detects rapid intensity changes; sharpen increases local contrast; blur kernels suppress noise; emboss highlights directional relief.

## 5. Built-in Keras convolution and pooling

**What:** Run a Keras `Conv2D` layer followed by `MaxPooling2D`.
**Why:** Built-in layers handle batched tensors and learn their kernels during training.

In [ ]:
builtin_conv = tf.keras.layers.Conv2D(
    filters=4, kernel_size=3, padding="same", activation="relu",
    kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED),
)
builtin_pool = tf.keras.layers.MaxPooling2D(pool_size=2)
builtin_input = tf.convert_to_tensor(x_train[:1])
builtin_features = builtin_conv(builtin_input)
builtin_pooled = builtin_pool(builtin_features)
print("Built-in structures:", builtin_input.shape, "->", builtin_features.shape, "->", builtin_pooled.shape)
assert tuple(builtin_features.shape) == (1, 28, 28, 4)
assert tuple(builtin_pooled.shape) == (1, 14, 14, 4)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for index, ax in enumerate(axes):
    ax.imshow(builtin_features[0, ..., index], cmap="viridis")
    ax.set_title(f"Built-in map {index + 1}")
    ax.axis("off")
plt.tight_layout()

## 6. CNN architecture and layer-by-layer structure

**What:** Build a CNN with two convolution/pooling stages, dropout, and dense classification.
**Why:** Convolutions add learned channels, pooling reduces spatial cost, and dropout regularizes the classifier.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1), name="input"),
    tf.keras.layers.Conv2D(8, 3, padding="same", activation="relu", name="conv1"),
    tf.keras.layers.MaxPooling2D(2, name="pool1"),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu", name="conv2"),
    tf.keras.layers.MaxPooling2D(2, name="pool2"),
    tf.keras.layers.Dropout(0.25, name="dropout"),
    tf.keras.layers.Flatten(name="flatten"),
    tf.keras.layers.Dense(64, activation="relu", name="dense"),
    tf.keras.layers.Dropout(0.25, name="dense_dropout"),
    tf.keras.layers.Dense(10, activation="softmax", name="classifier"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
print("Key structure: (28, 28, 1) -> (28, 28, 8) -> (14, 14, 8) -> (14, 14, 16) -> (7, 7, 16)")

### Shape trace

**What:** Forward a zero tensor through each layer and record the output shape.
**Why:** Explicit shape tracing catches channel and flattening mistakes before training.

In [ ]:
def trace_model_shapes(model, input_shape):
    tensor = tf.zeros((1, *input_shape), dtype=tf.float32)
    rows = [{"layer": "input", "output_shape": tuple(tensor.shape[1:]), "activations": int(np.prod(tensor.shape[1:]))}]
    for layer in model.layers:
        tensor = layer(tensor, training=False)
        rows.append({"layer": layer.name, "output_shape": tuple(tensor.shape[1:]), "activations": int(np.prod(tensor.shape[1:]))})
    return pd.DataFrame(rows)

shape_table = trace_model_shapes(model, (28, 28, 1))
display(shape_table)
assert tuple(shape_table.iloc[0].output_shape) == (28, 28, 1)
assert tuple(shape_table.loc[shape_table.layer == "pool1", "output_shape"].iloc[0]) == (14, 14, 8)
assert tuple(shape_table.loc[shape_table.layer == "pool2", "output_shape"].iloc[0]) == (7, 7, 16)

## 7. Complexity analysis

**What:** Count parameters, activation elements, and approximate convolution MACs.
**Why:** Parameters measure stored learnable capacity, while MACs estimate computation.

In [ ]:
def complexity_rows(model, input_shape):
    tensor = tf.zeros((1, *input_shape), dtype=tf.float32)
    rows = []
    for layer in model.layers:
        input_tensor = tensor
        tensor = layer(tensor, training=False)
        output_shape = tuple(int(dim) for dim in tensor.shape[1:])
        params = int(layer.count_params())
        macs = 0
        if isinstance(layer, tf.keras.layers.Conv2D):
            kh, kw = layer.kernel_size
            macs = int(np.prod(output_shape) * kh * kw * input_tensor.shape[-1])
        elif isinstance(layer, tf.keras.layers.Dense):
            macs = int(input_tensor.shape[-1] * layer.units)
        rows.append({"layer": layer.name, "output_shape": output_shape, "parameters": params, "activations": int(np.prod(output_shape)), "approx_MACs": macs})
    return pd.DataFrame(rows)

complexity_table = complexity_rows(model, (28, 28, 1))
display(complexity_table)
print("Total trainable parameters:", complexity_table.parameters.sum())
print("Keras model parameters:", model.count_params())
print("Total approximate MACs:", complexity_table.approx_MACs.sum())
assert complexity_table.parameters.sum() == model.count_params()
assert complexity_table.loc[complexity_table.layer == "conv1", "parameters"].iloc[0] == 8 * 3 * 3 * 1 + 8
assert complexity_table.loc[complexity_table.layer == "dense", "parameters"].iloc[0] == 7 * 7 * 16 * 64 + 64

## 8. Training with validation monitoring

**What:** Train with early stopping on validation loss and restore the best weights.
**Why:** Validation monitoring detects overfitting without using the test set for model selection.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=12, batch_size=128, callbacks=[early_stopping], verbose=2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()

## 9. Final test evaluation

**What:** Evaluate once on the untouched test set with class-balanced metrics.
**Why:** A confusion matrix and macro/weighted scores reveal errors that accuracy can hide.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
probabilities = model.predict(x_test, verbose=0)
y_pred = probabilities.argmax(axis=1)
assert len(y_pred) == len(y_test)
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
test_metrics = {"loss": float(test_loss), "accuracy": float(test_accuracy), "macro_f1": float(report["macro avg"]["f1-score"]), "weighted_f1": float(report["weighted avg"]["f1-score"])}
display(pd.DataFrame([test_metrics]))
display(report_df.round(3))
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(cm, cmap="Blues")
fig.colorbar(image, ax=ax)
ax.set(xlabel="Predicted label", ylabel="True label", title="MNIST confusion matrix")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
for row in range(10):
    for col in range(10):
        ax.text(col, row, cm[row, col], ha="center", va="center", fontsize=8)
plt.tight_layout()

## 10. Learned filters and feature maps

**What:** Visualize the first layer's learned kernels and feature maps for one test digit.
**Why:** Learned kernels adapt to training examples rather than following a fixed hand-designed rule.

In [ ]:
conv1 = model.get_layer("conv1")
learned_kernels = conv1.get_weights()[0]
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(learned_kernels[..., 0, index], cmap="coolwarm")
    ax.set_title(f"Kernel {index + 1}"); ax.axis("off")
plt.tight_layout()
feature_model = tf.keras.Model(model.input, conv1.output)
feature_maps = feature_model.predict(x_test[:1], verbose=0)[0]
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(feature_maps[..., index], cmap="viridis")
    ax.set_title(f"Feature map {index + 1}"); ax.axis("off")
plt.tight_layout()

## 11. Beginner-error audit

| Common mistake | Notebook fix |
|---|---|
| Data leakage | Split before transformations and keep test data isolated. |
| Test-set tuning | Use train/validation data for model decisions. |
| Ignoring overfitting | Plot validation curves and use early stopping. |
| Shape/channel errors | Assert input rank and print each layer structure. |
| Uncontrolled randomness | Seed Python, NumPy, and TensorFlow. |
| Accuracy-only evaluation | Report macro/weighted metrics and a confusion matrix. |
| Misreading filter output | Clip only for visualization; keep model inputs unchanged. |

**References:** [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html), [Google overfitting guidance](https://developers.google.com/machine-learning/crash-course/overfitting), [Keras Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/), and [Keras EarlyStopping](https://keras.io/api/callbacks/early_stopping/).